In [ ]:
from copy import deepcopy  # noqa: D100
import numpy as np
import xsdba
import xarray as xr
import xclim as xc
import xscen as xs

input_ref = '/project/ctb-frigon/sgammon/ESPOv2/reference/split_regions/IDM_CaSR_sr-0_default.zarr.zip'
input_rechunk = '/scratch/sgammon/espo-workdir/ScenarioMIP_BCC-CSM2-MR_ssp370+IDM+CaSR+sr-0+regchunked.zarr'
var = 'pr'

In [ ]:
ds_hist = xr.open_zarr(input_rechunk, decode_timedelta=False)
ds_ref = xr.open_zarr(input_ref, decode_timedelta=False)


In [ ]:
simcal = xc.core.calendar.get_calendar(ds_hist)
refcal = xs.utils.minimum_calendar(simcal, 'noleap')

In [ ]:
#ds_ref = xr.open_zarr(input_ref, decode_timedelta=False)
ds_ref = ds_ref.convert_calendar(refcal, align_on="year")

In [ ]:
import snakemake

In [ ]:
import yaml
from pathlib import Path
configpath = list(Path("../config").glob("*_ESPO.yml"))
config = [yaml.safe_load(open(p, 'r')) for p in configpath]
config = xs.config.recursive_update(*config)

In [ ]:
config

In [ ]:
import xsdba
import numpy as np

In [ ]:
ds_hist_c = ds_hist.compute()
ds_ref_c = ds_ref.compute()

In [ ]:
config

In [129]:
var = 'pr'
conf_train = deepcopy(config['biasadjust']['variables'][var]['training_args'])
if 'max_tail_factor' in conf_train['xsdba_train_args']:
    conf_train['xsdba_train_args'].pop('max_tail_factor')
#if 'group' in conf_train['xsdba_train_args']:
#    conf_train['xsdba_train_args'].pop('group')
#if 'adapt_freq_thresh' in conf_train['xsdba_train_args']:
#    conf_train['xsdba_train_args'].pop('adapt_freq_thresh')


In [ ]:
ds_tr_c = xs.train(
    dref=ds_ref_c,
    dhist=ds_hist_c,#.isel(realization=0),
    var=[var],
    **conf_train
)

In [130]:
conf_train

{'method': 'DetrendedQuantileMapping',
 'period': ['1989', '2023'],
 'group': {'group': 'time.dayofyear',
  'window': 31,
  'add_dims': ['realization']},
 'jitter_under': {'thresh': '1.1574e-07 kg m^-2 s^-1'},
 'xsdba_train_args': {'kind': '*',
  'nquantiles': 50,
  'adapt_freq_thresh': '1.1574e-05 kg m^-2 s^-1'}}

In [145]:
xsdba.Grouper.from_kwargs(**conf_train['group'])['group']

Grouper(add_dims=['realization'], name='time.dayofyear', window=31)

In [146]:
xsdba.__version__

'0.6.1'

In [144]:
print(Path(xsdba._processing.__file__).read_text())

"""
Compute Functions Submodule

Here are defined the functions wrapped by map_blocks or map_groups.
The user-facing, metadata-handling functions should be defined in processing.py.
"""

from __future__ import annotations
from collections.abc import Sequence

import numpy as np
import xarray as xr

from xsdba import nbutils as nbu
from xsdba.base import Grouper, map_groups
from xsdba.utils import ADDITIVE, apply_correction, ecdf, invert, rank


@map_groups(
    sim_ad=[Grouper.ADD_DIMS, Grouper.DIM],
    dP0=[Grouper.PROP],
    P0_ref=[Grouper.PROP],
    P0_hist=[Grouper.PROP],
    pth=[Grouper.PROP],
)
def _adapt_freq(ds: xr.Dataset, *, dim: Sequence[str], thresh: float = 0, kind: str = "+") -> xr.Dataset:
    r"""
    Adapt frequency of values under thresh of `sim`, in order to match ref.

    This is the compute function, see :py:func:`xsdba.processing.adapt_freq` for the user-facing function.

    Parameters
    ----------
    ds : xr.Dataset
        Dataset variables:
            

In [133]:
xsdba.processing.adapt_freq(
    ref=ds_ref_c['pr'], 
    sim=ds_hist_c['pr'], 
    thresh='1.1574e-05 kg m^-2 s^-1', 
    **xsdba.Grouper.from_kwargs(**conf_train['group'])
)

ValueError: group__adapt_freq failed on block with coords : Coordinates:
  * loc          (loc) int64 360B 0 1 2 3 4 5 6 7 8 ... 37 38 39 40 41 42 43 44
  * realization  (realization) object 8B 'CMIP6_ScenarioMIP_BCC_BCC-CSM2-MR_s...
  * time         (time) object 438kB 1951-01-01 00:00:00 ... 2100-12-31 00:00:00.

In [128]:
config['biasadjust']['variables']['pr']['training_args']['xsdba_train_args']

{'kind': '*',
 'nquantiles': 50,
 'adapt_freq_thresh': '1.1574e-05 kg m^-2 s^-1',
 'max_tail_factor': 10}

In [ ]:
ds_tr_c = ds_tr.compute()

In [ ]:
from workflow.scripts.utils import dask_cluster


if 1 == 0:  # trick vscode
    import snakemake


if __name__ == '__main__':

    # Get Snakemake parameters
    var = snakemake.wildcards.var
    input_ref = snakemake.input.ref
    input_rechunk = snakemake.input.rechunk
    output = snakemake.output[0]
    config = deepcopy(snakemake.config)

    client = dask_cluster(snakemake.params, config['dask']['client'])

    # load hist ds (simulation)
    ds_hist = xr.open_zarr(input_rechunk, decode_timedelta=False)

    if 'hursmin' in ds_hist:
        # trick for biasadjustement of hursmin (sim) on hursTasmax (ref)
        ds_hist = ds_hist.rename({'hursmin': 'hursTasmax'})

    # load ref ds
    # choose right calendar
    simcal = xc.core.calendar.get_calendar(ds_hist)
    refcal = xs.utils.minimum_calendar(simcal, 'noleap')

    ds_ref = xr.open_zarr(input_ref, decode_timedelta=False)
    ds_ref = ds_ref.convert_calendar(refcal, align_on="year")
    
    if var == "pr":
        context = "hydro"
    else:
        context = "none"
    with xc.units.units.context(context):
        # training
        ds_tr = xs.train(
            dref=ds_ref,
            dhist=ds_hist,
            var=[var],
            **config['biasadjust']['variables'][var]['training_args']
        )

    xs.save_to_zarr(
        ds_tr,
        output,
        rechunk=config['chunks']['workingloc'],
        **config['save_to_zarr'],
    )